# Feedback Prize - Predicting Effective Arguments

## Data understanding

In [1]:
!ls ../input/feedback-prize-effectiveness

## An example of essay

In [2]:
cat ../input/feedback-prize-effectiveness/train/00066EA9880D.txt

In [3]:
import pandas as pd

input_dir = "../input/feedback-prize-effectiveness"
train_df = pd.read_csv(f"{input_dir}/train.csv")
train_df.head()

### Ratings distribution by discourse type

In [4]:
import matplotlib.pyplot as plt

discourse_types = ["Lead", "Position", "Claim", "Counterclaim", "Rebuttal", "Evidence", "Concluding Statement"]
fig, axs = plt.subplots(7, 1)

axs = axs.flatten()
for i, discourse_type in enumerate(discourse_types):
    axs[i] = (train_df.loc[train_df.discourse_type == discourse_type]
              .discourse_effectiveness.value_counts().plot(kind='barh', ax=axs[i], figsize=(20, 30)))
    axs[i].set_title(f"{discourse_type} ratings distribution", fontsize=16)
    axs[i].bar_label(axs[i].containers[0], label_type="edge")
    
plt.tight_layout()
plt.show()

In all discourse types the most frequent evaluation is "Adequate"!

### Average discourse length by rating

In [5]:
train_df["discourse_length"] = train_df.discourse_text.map(lambda x: len(x))
train_df["words_number"] = train_df.discourse_text.map(lambda x: len(x.split()))
ax = train_df.groupby("discourse_effectiveness").agg({"discourse_length": "mean", "words_number": "mean"}).plot(kind="bar")
ax.set_title("Discourse length and words number by rating", fontsize=16)
plt.show()

Effective arguments are the longest!

### Unique and repeated words

In [6]:
from collections import Counter

train_df["unique_words_percentage"] = train_df.discourse_text.map(lambda x: len(set(x.split()))) / train_df.words_number

train_df["repeated_words_percentage"] = train_df.discourse_text.map(
    lambda x: len([value for value in Counter(x.split()).values() if value > 1])) / train_df.words_number
ax = (
    train_df.groupby("discourse_effectiveness").agg({"unique_words_percentage": "mean", "repeated_words_percentage": "mean"})
    .plot(kind='barh'))
ax.set_title("Discourse unique words and repeated words by rating", fontsize=16)
plt.show()

## Text preprocessing

In [ ]:
!pip install spacy-transformers

In [ ]:
!python -m spacy download "en_core_web_trf"

In [ ]:
import string
import spacy
from spacy.lang.en import English

import nltk

nlp = spacy.load("en_core_web_trf", disable=['ner', 'parser', 'lemmatizer', 'textcat'])
parser = English()

def punctuation_removal(tokens):
    punctuations = string.punctuation
    tokens = [token for token in tokens if token not in punctuations]
    return tokens

def stopwords_removal(tokens):
    stopwords = spacy.lang.en.stop_words.STOP_WORDS
    tokens = [token for token in tokens if token not in stopwords]
    return tokens

def stemming(tokens):
    porter = nltk.PorterStemmer()
    stems = [porter.stem(token) for token in tokens]
    return stems
    
def lemmatization(tokens):
    wordnet_lemmatizer = nltk.WordNetLemmatizer()
    lemmas = [wordnet_lemmatizer.lemmatize(token) for token in tokens]
    return lemmas
    
def spacy_tokenizer(phrase, steps=[punctuation_removal]):
    phrase = phrase.strip().lower()
    tokens = parser(phrase)
    tokens = [token.text for token in tokens]
    
    for step in steps:
        tokens = step(tokens)
    
    return tokens

In [ ]:
train_df["discourse_tokens"] = train_df.discourse_text.map(lambda x: spacy_tokenizer(x))

### TF - IDF

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = train_df.discourse_tokens
vectorizer = TfidfVectorizer(analyzer='word', tokenizer=lambda x: x, preprocessor=lambda x: x, token_pattern=None)
vectors = vectorizer.fit_transform(corpus)
vocabulary = vectorizer.get_feature_names_out()

In [ ]:
train_df = pd.concat([train_df, pd.DataFrame(vectors)[0].map(lambda x: x)], axis=1)

In [ ]:
train_df = train_df.rename(columns={0: 'tf_idf_vector'})
train_df

In [ ]:
def get_tf_idf_dict(tf_idf_vector):
    tf_idf_vector = tf_idf_vector.toarray().squeeze()
    nonzero = tf_idf_vector.nonzero()
    return dict(zip(vocabulary[nonzero], tf_idf_vector[nonzero]))

train_df["tf_idf_dict"] = train_df.tf_idf_vector.map(get_tf_idf_dict)

### Embeddings

In [ ]:
import gensim
import gensim.downloader

model_w2v = gensim.downloader.load("word2vec-google-news-300")

In [ ]:
import numpy as np

train_df["embeddings"] = train_df.tf_idf_dict.map(
    lambda dict: [model_w2v[key] * value for (key, value) in dict.items() if key in model_w2v])

In [ ]:
train_df["embeddings_number"] = train_df.embeddings.map(len)
train_df = train_df.loc[train_df.embeddings_number > 0]

In [ ]:
train_df

In [ ]:
train_df["embeddings_matrix"] = train_df.embeddings.map(lambda embeddings: np.array(embeddings))

In [ ]:
train_df["doc_embedding"] = train_df.embeddings_matrix.map(lambda m: m.mean(axis=0))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_similarity([train_df.iloc[0].doc_embedding, train_df.iloc[1].doc_embedding])

In [ ]:
tmp = train_df.loc[train_df.essay_id == "FFA381E58FC6"]

In [ ]:
essay = tmp.copy()
essay = essay.loc[:, ['discourse_id', 'discourse_type', 'essay_id', 'doc_embedding']]
essay = essay.merge(essay, on="essay_id")
essay = essay.loc[essay.discourse_type_x == "Evidence"]
essay = essay.loc[
    (essay.discourse_type_y == 'Claim') |
    (essay.discourse_type_y == 'Counterclaim') |
    (essay.discourse_type_y == 'Rebuttal')]

In [ ]:
def compute_similarity(row):
    return cosine_similarity([row.doc_embedding_x, row.doc_embedding_y])[0, 1]

essay = pd.concat([essay, essay.apply(compute_similarity, axis=1)], axis=1)

In [ ]:
essay = essay.reset_index(drop=True)

In [ ]:
essay